# 节点 3：动态天气与晾晒判断

这一节点让页面中的日期和天气随现实变化，并把天气与物品到期状态结合。

## 1. 本节点目标

用户输入城市后，系统先解析经纬度，再查询当日预报，最后转换成统一结构。晴天且物品到期时显示愤怒表情；天气失败时仍按周期提醒。

## 2. 完成结果与验收

- 使用免密钥 Open-Meteo 查询城市与今日天气。
- 展示当前温度、最高最低温、降雨概率、湿度和风速。
- 相同城市缓存 30 分钟，也可手动刷新。
- 无效城市、超时和缺字段会变成中文错误并降级。
- mock 测试不访问真实网络；真实北京天气已验证成功。
- 首页使用低饱和蓝、浅灰、白色主题。

## 3. 本节点文件结构

```text
src/smart_laundry/weather.py   城市解析、天气转换与错误映射
app.py                         缓存、天气卡片、晴天提醒和主题
tests/test_weather.py          固定响应、无效城市和超时测试
notebooks/04_weather.ipynb     本学习说明
```

## 4. 关键代码解释

`WeatherService.get_today()` 先调用城市解析接口获得经纬度，再请求预报接口。外部 JSON 不直接进入 UI，而是转换为 `WeatherSummary`。这样字段缺失会在天气模块内被发现。

页面使用 `@st.cache_data(ttl=1800)`：1800 秒内同一城市复用结果，减少网络等待；“刷新天气”会主动清空缓存。

`is_sunny` 只根据明确的晴朗天气代码判断；`is_suitable_for_drying` 还会检查降雨概率和湿度，所以“晴”不一定等于最适合晾晒。

In [ ]:
weather_code = 0
rain_probability = 10
humidity = 55
is_sunny = weather_code in (0, 1)
is_suitable = weather_code <= 2 and rain_probability < 40 and humidity < 85
print(is_sunny, is_suitable)

## 5. 数据流

```mermaid
flowchart LR
 A[城市名称] --> B[Open-Meteo Geocoding]
 B --> C[经纬度]
 C --> D[Forecast API]
 D --> E[WeatherSummary]
 E --> F[晾晒适宜判断]
 F --> G[首页天气区]
 E --> H{晴天且物品到期?}
 H -->|是| I[😠 醒目提醒]
 H -->|否| J[普通状态表情]
```

## 6. 关键概念

- **API**：程序之间约定好的数据接口。
- **Geocoding**：把城市名称转换为经纬度。
- **JSON**：天气服务常用的数据格式。
- **timeout**：网络等待超过上限后停止。
- **cache TTL**：缓存可以保留的时间。
- **provider abstraction**：页面不依赖外部服务的原始格式。
- **fallback**：外部服务失败时继续提供基本能力。

## 7. 为什么这样设计

首版选择免密钥服务，降低配置门槛；代价是依赖公共网络和第三方可用性。因此天气模块设置超时、统一错误和短缓存，页面始终保留只按周期运行的降级路径。天气只影响建议强度，不改变数据库事实。

## 8. 常见错误与排查

1. **找不到城市**：输入更具体的“城市, 省份/国家”。
2. **天气一直加载**：检查网络，等待超时后页面会自动降级。
3. **天气没有更新**：点击刷新，或等待 30 分钟缓存到期。
4. **字段不完整**：不要在页面直接索引原始 JSON，检查转换层错误。
5. **测试偶尔失败**：单元测试必须使用 FakeSession，不能依赖真实天气。

## 9. 面试可能追问

**问：为什么天气要先解析经纬度？** 答：预报接口使用坐标，城市名可能重名，解析层负责选择标准位置。

**问：如何处理 API 不稳定？** 答：超时、错误映射、缓存、mock 测试和规则降级。

**问：为什么晴天还可能不适合晾晒？** 答：晾晒判断还考虑降雨概率和湿度，规则会明确展示依据。

## 10. 必须掌握的最少知识

需要理解一次天气查询包含城市解析和预报两步；外部 JSON 必须转换校验；网络可能失败；缓存减少重复请求；规则模式保证页面不会因天气失败而瘫痪。

## 11. 可自测小题

1. 为什么天气接口需要经纬度？
2. TTL 1800 表示多久？
3. 为什么测试不能调用真实天气？
4. 天气失败后页面还能做什么？

<details><summary>参考答案</summary>

1. 坐标能稳定定位预报位置。2. 30 分钟。3. 网络和天气结果不稳定。4. 继续按物品周期计算并提醒。

</details>

## 12. 动手小练习

1. 修改上方湿度为 90，观察晾晒适宜结果。
2. 在页面切换两个城市，比较降雨概率。
3. 给天气测试增加一个雨天固定响应。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| latitude/longitude | 纬度和经度 |
| forecast | 对未来天气的预报 |
| weather code | 标准天气现象编号 |
| humidity | 空气相对湿度 |
| precipitation probability | 降水概率 |
| cache | 临时保存可复用结果 |
| mock | 测试中替代真实外部服务 |

## 14. 下一节点连接

下一节点会在已经结构化的物品状态和天气之上生成规则建议，再封装可选 LLM provider。即使没有模型密钥，也继续使用当前确定性数据生成基础计划。